# Interactive graph visualization

Staged views of a measurement-based CV computation with `photographiq.interactive`.
Python is the authoritative model: every number below is computed by PhotoGraphiQ; the
browser (Cytoscape.js / Plotly.js from pinned CDNs) only displays and edits.
Pages are shown inline as `srcdoc` iframes; `.show()` opens them in a browser and
`.save_html(path)` writes a standalone file. Static figures use Matplotlib only.

In [ ]:
import json
import tempfile
from pathlib import Path

import photographiq as pg
from photographiq.cvflow import flow_pattern
from photographiq.flows import analyze
from photographiq.interactive import (
    GraphView, apply_edits, covariance_heatmap, execution_trace, squeezing_ellipses, wigner_plot,
)
from photographiq.ir import OpenGraph

graph = pg.CVGraph.line(4, squeezing=1.5, inputs=(0,), outputs=(3,))
shears = {0: 0.3, 1: -0.5, 2: 0.0}
open_graph = OpenGraph.from_cvgraph(graph, shears)
flow_result = analyze(open_graph)  # CV flow search + certificate + determinism check
pattern = flow_pattern(graph, flow_result.measurement_order, shears=shears)
result = pg.simulate(pattern, backend="gaussian", seed=7)
result.outcomes

## Stage A: construct (graph, weights, roles, squeezing; no flow arrows)

In [ ]:
view = GraphView(open_graph, title="Four-node CV wire").construct()
view

## Stage B: flow overlay

`with_flow` takes the `photographiq.flows.FlowAnalysisResult` (or any object/dict with the
same attributes): certification status, algorithm, assumptions, measurement order,
layers and dependency arrows derived from its `correction_map`.

In [ ]:
print(flow_result.status, flow_result.algorithm, flow_result.measurement_order)
flow_view = view.with_flow(flow_result)
sorted(flow_view.flow_arrows())

In [ ]:
flow_view

## Stage C: corrections (displacements as linear combinations of outcomes m_k)

In [ ]:
correction_view = flow_view.with_corrections()
for target, correction in correction_view.corrections.items():
    print(target, correction.formula())
correction_view

## Stage D: execution with the recorded outcomes (Play / Step / Reset)

In [ ]:
trace = execution_trace(pattern, result)
for step in trace.steps:
    print(step.index, step.kind, step.label)
execution_view = correction_view.with_execution(trace)
execution_view

## Static manuscript export and the edit round trip

In [ ]:
out = Path(tempfile.mkdtemp())
for stage in ("construct", "flow", "corrections", "execution"):
    execution_view.save(out / f"wire_{stage}.svg", stage=stage)
execution_view.save_html(out / "wire.html")
# The page's Export button produces exactly this structure; Python re-validates it.
exported = json.dumps({"graph": view.graph.to_dict(), "layout": view.layout.to_dict()})
edited, layout = apply_edits(exported)
assert edited.to_dict() == view.graph.to_dict()
sorted(p.name for p in out.iterdir())

## Scientific plots of the output state

In [ ]:
state = result.state
covariance_heatmap(state)

In [ ]:
squeezing_ellipses(state)

In [ ]:
wigner_plot(state, 3, points=81, extent=8)